# Field Concept Evaluation: From Definition to Multi-Criterion Assessment


In [1]:
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np


def find_project_root():
    env_root = os.environ.get("NEQSIM_PROJECT_ROOT")
    candidates = []
    if env_root:
        candidates.append(Path(env_root).resolve())
    try:
        candidates.append(Path(__vsc_ipynb_file__).resolve())
    except NameError:
        candidates.append(Path.cwd().resolve())
    expanded = []
    for candidate in candidates:
        expanded.extend([candidate] + list(candidate.parents))
    for candidate in expanded:
        if (candidate / "pom.xml").exists() and (candidate / "devtools" / "neqsim_dev_setup.py").exists():
            return candidate
    raise RuntimeError("Could not find NeqSim project root")


try:
    NOTEBOOK_DIR = Path(__vsc_ipynb_file__).resolve().parent
except NameError:
    NOTEBOOK_DIR = Path.cwd().resolve()

FIGURES_DIR = NOTEBOOK_DIR.parent / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
PROJECT_ROOT = find_project_root()
sys.path.insert(0, str(PROJECT_ROOT / "devtools"))

NEQSIM_AVAILABLE = False
NEQSIM_ERROR = ""
try:
    from neqsim_dev_setup import neqsim_init

    ns = neqsim_init(project_root=PROJECT_ROOT, recompile=False, verbose=False)
    JClass = ns.JClass
    NEQSIM_AVAILABLE = True
except Exception as exc:
    ns = None
    JClass = None
    NEQSIM_ERROR = str(exc)

print(f"Notebook directory: {NOTEBOOK_DIR}")
print(f"Figures directory: {FIGURES_DIR}")
print(f"NeqSim Java bridge available: {NEQSIM_AVAILABLE}")
if NEQSIM_ERROR:
    print(f"NeqSim bridge warning: {NEQSIM_ERROR}")


Notebook directory: C:\Users\ESOL\Documents\GitHub\neqsim\neqsim-paperlab\books\tpg4230_field_development_and_operations_2026\chapters\ch11_field_development_building_blocks\notebooks
Figures directory: C:\Users\ESOL\Documents\GitHub\neqsim\neqsim-paperlab\books\tpg4230_field_development_and_operations_2026\chapters\ch11_field_development_building_blocks\figures
NeqSim Java bridge available: True


## Concept Alternatives and Evaluation Criteria


In [2]:
concepts = ["Tieback", "Standalone platform", "FPSO redeploy"]
criteria = ["NPV", "CAPEX", "Flow assurance", "Schedule", "Emissions"]
weights = np.array([0.40, 0.22, 0.16, 0.12, 0.10])
scores = np.array([[78, 82, 66, 86, 74], [84, 45, 72, 52, 56], [70, 58, 62, 68, 60]])
weighted = scores * weights
total = weighted.sum(axis=1)
for concept, value in sorted(zip(concepts, total), key=lambda row: row[1], reverse=True):
    print(f"{concept:<20} weighted score = {value:5.1f}")


Tieback              weighted score =  77.5
Standalone platform  weighted score =  66.9
FPSO redeploy        weighted score =  64.8


## Weighted Ranking


In [3]:
order = np.argsort(total)
fig, ax = plt.subplots(figsize=(9, 5))
ax.barh(np.array(concepts)[order], total[order], color=["#79addc", "#ffc857", "#9bc53d"])
ax.set_xlabel("Weighted score (0-100)")
ax.set_title("Concept Ranking by Multi-Criterion Decision Analysis")
ax.grid(axis="x", alpha=0.3)
for i, idx in enumerate(order):
    ax.text(total[idx] + 1, i, f"{total[idx]:.1f}", va="center")
fig.savefig(FIGURES_DIR / "ch11_concept_mcda_ranking.png", dpi=150, bbox_inches="tight")
plt.close(fig)


**Discussion (Figure ch11_concept_mcda_ranking.png).** *Observation.* The tieback option has the highest weighted score despite not having the highest standalone NPV score. *Mechanism.* Lower CAPEX, faster schedule and lower emissions offset the larger absolute development potential of the standalone platform. *Implication.* A single economic metric can hide execution and operability advantages. *Recommendation.* Present both raw criteria and weighted totals at DG1/DG2 so decision makers see the trade-offs.


## Score Heatmap


In [4]:
fig, ax = plt.subplots(figsize=(9.5, 4.8))
image = ax.imshow(scores, cmap="YlGnBu", vmin=0, vmax=100)
ax.set_xticks(range(len(criteria)), labels=criteria, rotation=25, ha="right")
ax.set_yticks(range(len(concepts)), labels=concepts)
for i in range(scores.shape[0]):
    for j in range(scores.shape[1]):
        ax.text(j, i, f"{scores[i, j]:.0f}", ha="center", va="center", color="black")
ax.set_title("Concept Evaluation Criteria Scores")
fig.colorbar(image, ax=ax, label="Score (higher is better)")
fig.savefig(FIGURES_DIR / "ch11_concept_score_heatmap.png", dpi=150, bbox_inches="tight")
plt.close(fig)


**Discussion (Figure ch11_concept_score_heatmap.png).** *Observation.* The heatmap shows why each concept scores differently, not only which concept wins. *Mechanism.* Scores are normalized to the same preference direction, so high values consistently mean desirable outcomes. *Implication.* This format reduces ambiguity when cost, risk and value are combined. *Recommendation.* Always keep the normalized criteria matrix with the concept-selection record.


## Exercises

1. Recalculate the ranking with the CAPEX weight increased to 0.35.
2. Which criterion most strongly differentiates the tieback and standalone platform?
3. Explain why schedule risk should not be hidden inside CAPEX contingency.
4. Add a safety-readiness criterion and propose a defensible weight.
5. State one weakness of weighted-sum MCDA for sanction decisions.
